# Stability of the Balanced State: Lin (bilinear) vs Saturating Model

**Claim to be demonstrated:** the saturating model has a *stable* balanced
(healthy) equilibrium; the bilinear Lotka--Volterra model of Lin & Han does not.

Four independent lines of evidence:
1. **Equilibria** — closed forms for both models.
2. **Eigenvalues** — plotted in the complex plane; stability = all in the left half.
3. **Perturbation recovery** — nudge one organ and watch what happens (the physical test).
4. **Stability maps** — where in parameter space each model is stable.

Run top to bottom (Runtime -> Run all). ~1-2 minutes, nothing to upload.


In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

NAMES=["Wood","Fire","Earth","Metal","Water"]
COLORS=["green","red","magenta","black","blue"]


## 1. The two models

Same graph, different coupling forms. Index maps via `np.roll`:
mother `i-1`, controller `i-2`, child `i+1`, target `i+2`.

**Lin (bilinear):** $\dot x_i = ax_i-bx_i^2+cx_ix_{i-1}-dx_ix_{i-2}-gx_ix_{i+1}-hx_ix_{i+2}$

**Saturating (ours):** $\dot x_i = ax_i-kx_i^2+\frac{cx_{\sigma(i)}}{1+x_i}-\frac{gx_i}{1+x_{\kappa(i)}}-hx_ix_{\rho(i)}$


In [ ]:
# ---------- LIN bilinear ----------
def rhs_lin(t,x,a,b,c,d,g,h):
    x=np.asarray(x)
    return (a*x-b*x**2 + c*x*np.roll(x,1) - d*x*np.roll(x,2)
            - g*x*np.roll(x,-1) - h*x*np.roll(x,-2))

def eq_lin(a,b,c,d,g,h):
    """Lin's symmetric equilibrium a/(b+d+g+h-c) (their eq. 7)."""
    M=b+d+g+h-c
    return a/M if M>0 else np.nan

def jac_lin(xv,a,b,c,d,g,h):
    J=np.zeros((5,5))
    for i in range(5):
        im1,im2,ip1,ip2=(i-1)%5,(i-2)%5,(i+1)%5,(i+2)%5
        J[i,i]=a-2*b*xv[i]+c*xv[im1]-d*xv[im2]-g*xv[ip1]-h*xv[ip2]
        J[i,im1]+=c*xv[i]; J[i,im2]+=-d*xv[i]
        J[i,ip1]+=-g*xv[i]; J[i,ip2]+=-h*xv[i]
    return J

# ---------- SATURATING (ours) ----------
def rhs_sat(t,x,a,k,c,g,h):
    x=np.asarray(x)
    return (a*x-k*x**2 + c*np.roll(x,1)/(1+x)
            - g*x/(1+np.roll(x,-2)) - h*x*np.roll(x,2))

def eq_sat(a,k,c,g,h):
    """Positive root of (k+h)x^2 + [(k+h)-a]x - (a+c-g) = 0."""
    A=k+h; B=(k+h)-a; C=-(a+c-g)
    disc=B*B-4*A*C
    if disc<0: return np.nan
    r=(-B+np.sqrt(disc))/(2*A)
    return r if r>0 else np.nan

def jac_sat(xv,a,k,c,g,h):
    J=np.zeros((5,5))
    for i in range(5):
        im1,ip2,im2=(i-1)%5,(i+2)%5,(i-2)%5
        mo,ta,co=xv[im1],xv[ip2],xv[im2]
        J[i,i]=a-2*k*xv[i]-c*mo/(1+xv[i])**2-g/(1+ta)-h*co
        J[i,im1]+=c/(1+xv[i])
        J[i,ip2]+=g*xv[i]/(1+ta)**2
        J[i,im2]+=-h*xv[i]
    return J


## 2. Equilibria and eigenvalues

Baseline parameters: Lin at $a=b=c=d=g=h=1$ (their Case 1);
ours at $a=2,k=0.5,c=1,g=0.5,h=0.3$ (the table in our paper).


In [ ]:
P_LIN=(1,1,1,1,1,1)
P_SAT=(2,0.5,1,0.5,0.3)

xl=eq_lin(*P_LIN); Jl=jac_lin(np.full(5,xl),*P_LIN); el=np.linalg.eigvals(Jl)
xs=eq_sat(*P_SAT); Js=jac_sat(np.full(5,xs),*P_SAT); es=np.linalg.eigvals(Js)

print(f"LIN  x* = {xl:.6f}   (their reported 1/3)")
for e in sorted(el,key=lambda z:-z.real): print(f"     {e.real:+.4f}{e.imag:+.4f}i")
print(f"     max Re = {max(el.real):+.4f}  ->  {'UNSTABLE' if max(el.real)>0 else 'stable'}")

print(f"\nSAT  x* = {xs:.6f}   (closed form 3/4+sqrt(59)/4 = {0.75+np.sqrt(59)/4:.6f})")
for e in sorted(es,key=lambda z:-z.real): print(f"     {e.real:+.4f}{e.imag:+.4f}i")
print(f"     max Re = {max(es.real):+.4f}  ->  {'unstable' if max(es.real)>0 else 'STABLE'}")

print("\nJacobian of our model is circulant:",
      all(np.allclose(Js[i],np.roll(Js[0],i)) for i in range(5)))
w=np.exp(2j*np.pi/5); r0=Js[0]
lam=np.array([sum(r0[m]*w**(j*m) for m in range(5)) for j in range(5)])
print("closed-form DFT eigenvalues agree with numpy:",
      np.allclose(np.sort_complex(lam),np.sort_complex(es)))


### Eigenvalue plot — the stability figure

Stability means **every** eigenvalue lies strictly left of the imaginary axis.


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
ax.axvline(0,color='k',lw=1)
ax.axhline(0,color='gray',lw=0.5,ls=':')
ax.fill_betweenx([-1.5,1.5],0,3,color='red',alpha=0.06)
ax.text(0.35,1.25,'unstable half-plane',color='red',fontsize=9)

ax.scatter(el.real,el.imag,s=110,marker='o',facecolor='none',
           edgecolor='crimson',lw=2,label='Lin (bilinear)')
ax.scatter(es.real,es.imag,s=110,marker='x',color='navy',lw=2,
           label='Saturating (ours)')

ax.set_xlabel(r'Re $\lambda$'); ax.set_ylabel(r'Im $\lambda$')
ax.set_title('Eigenvalues of the Jacobian at the balanced state')
ax.legend(loc='lower left'); ax.set_xlim(-2.6,0.9); ax.set_ylim(-1.4,1.4)
ax.grid(alpha=0.25); plt.tight_layout(); plt.show()

print("Lin has two eigenvalues in the red (unstable) region; ours has none.")


## 3. Perturbation recovery — the physical test

Eigenvalues are a local statement. Here we test it directly: start at the balanced
state, nudge **one** organ, and integrate.

- **Small perturbation:** +5% on Wood.
- **Severe perturbation:** Wood reduced to 30% of balance (a deficiency pattern).

A stable model returns to balance; an unstable one does not.


In [ ]:
def perturb(rhs,params,xstar,mode="small",t_end=120):
    x0=np.full(5,xstar)
    if mode=="small": x0[0]+=0.05*xstar
    else:             x0[0]*=0.30
    sol=solve_ivp(rhs,(0,t_end),x0,args=params,method="RK45",
                  t_eval=np.linspace(0,t_end,3000),rtol=1e-9,atol=1e-11)
    dev=np.linalg.norm(sol.y-xstar,axis=0)
    return sol,dev

fig,axes=plt.subplots(2,2,figsize=(13,7))
for col,(nm,rhs,params,xstar) in enumerate(
        [("LIN (bilinear)",rhs_lin,P_LIN,xl),
         ("SATURATING (ours)",rhs_sat,P_SAT,xs)]):
    for row,mode in enumerate(["small","severe"]):
        sol,dev=perturb(rhs,params,xstar,mode)
        ax=axes[row,col]
        for i in range(5):
            ax.plot(sol.t,sol.y[i],color=COLORS[i],label=NAMES[i],lw=1.1)
        ax.axhline(xstar,color='gray',ls='--',lw=1)
        ax.set_title(f"{nm} — {mode} perturbation")
        ax.set_xlabel("time"); ax.set_ylabel("$x_i$")
        if row==0 and col==0: ax.legend(ncol=5,fontsize=7)
plt.tight_layout(); plt.show()

# quantitative summary
print(f"{'':22s}{'start dev':>10s}{'final dev':>12s}{'verdict':>14s}")
for nm,rhs,params,xstar in [("LIN small",rhs_lin,P_LIN,xl),
                            ("LIN severe",rhs_lin,P_LIN,xl),
                            ("SAT small",rhs_sat,P_SAT,xs),
                            ("SAT severe",rhs_sat,P_SAT,xs)]:
    mode="small" if "small" in nm else "severe"
    sol,dev=perturb(rhs,params,xstar,mode)
    verdict="recovers" if dev[-1]<1e-3 else "does NOT recover"
    print(f"{nm:22s}{dev[0]:10.4f}{dev[-1]:12.4f}{verdict:>18s}")


The dashed grey line is the balanced state. In the Lin panels the organs leave it
and never come back — the small perturbation *grows* and the severe one collapses
organs toward zero. In our panels the trajectories return to the line in both cases.


## 4. Stability maps over parameter space

For each $(c,h)$ we recompute the equilibrium, its Jacobian, and
$\max_j \operatorname{Re}\lambda_j$. Blue = stable, red = unstable, and the black
contour is the **stability boundary** $\max\operatorname{Re}\lambda = 0$ — the
model's picture of the transition out of health.


In [ ]:
N=90
c_vals=np.linspace(0.05,2.5,N)
h_vals=np.linspace(0.05,2.0,N)

Mlin=np.full((N,N),np.nan); Msat=np.full((N,N),np.nan)
for iy,hv in enumerate(h_vals):
    for ix,cv in enumerate(c_vals):
        # Lin: vary c and h, others = 1
        xe=eq_lin(1,1,cv,1,1,hv)
        if np.isfinite(xe) and xe>0:
            Mlin[iy,ix]=max(np.linalg.eigvals(jac_lin(np.full(5,xe),1,1,cv,1,1,hv)).real)
        # Ours: vary c and h, a=2,k=0.5,g=0.5
        xe2=eq_sat(2,0.5,cv,0.5,hv)
        if np.isfinite(xe2) and xe2>0:
            Msat[iy,ix]=max(np.linalg.eigvals(jac_sat(np.full(5,xe2),2,0.5,cv,0.5,hv)).real)

fig,axes=plt.subplots(1,2,figsize=(13,5))
vmax=np.nanmax(np.abs([np.nanmin([Mlin,Msat]),np.nanmax([Mlin,Msat])]))
for ax,M,title in [(axes[0],Mlin,"Lin (bilinear)"),(axes[1],Msat,"Saturating (ours)")]:
    im=ax.imshow(M,origin="lower",aspect="auto",cmap="RdBu_r",
                 vmin=-vmax,vmax=vmax,
                 extent=[c_vals[0],c_vals[-1],h_vals[0],h_vals[-1]])
    CS=ax.contour(c_vals,h_vals,M,levels=[0],colors='k',linewidths=2)
    ax.clabel(CS,fmt={0:'stability boundary'},fontsize=8)
    ax.set_xlabel("c  (Sheng / generation strength)")
    ax.set_ylabel("h  (Ke restraint strength)")
    frac=100*np.nanmean(M<0)
    ax.set_title(f"{title}\nstable over {frac:.0f}% of the region")
    fig.colorbar(im,ax=ax,label=r"max Re $\lambda$")
plt.suptitle("Stability of the balanced state (blue = stable)")
plt.tight_layout(); plt.show()


## 5. Summary

| | Lin (bilinear) | Saturating (ours) |
|---|---|---|
| balanced equilibrium | $a/(b+d+g+h-c)$ | positive root of $(k+h)x^2+[(k+h)-a]x-(a+c-g)$ |
| at baseline | $x^*=1/3$ | $x^*=\tfrac34+\tfrac{\sqrt{59}}4\approx2.6703$ |
| max Re $\lambda$ | $+0.206$ | $-1.154$ |
| verdict | **unstable** | **stable** |
| small perturbation | grows | decays |
| severe perturbation | organs collapse toward 0 | full recovery |

**Interpretation for the paper.** Health is a state the body *returns to* after a
disturbance — an asymptotically stable equilibrium. The bilinear model cannot
represent this at its symmetric state, so it has no mathematical notion of a healthy
baseline; the saturating model does. The difference traces to a single modeling
choice: replacing unbounded mass-action couplings with saturating, demand-limited
ones.
